In [2]:
import os
import librosa
import numpy as np
import pandas as pd

In [4]:
emotion_dict = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'afraid',
    '07': 'disgust',
    '08': 'surprised'
}

In [5]:
paths = []
emotions = []

dataset_path = r"C:\Users\saniy\Downloads\code_Alpha_T2"

for actor in os.listdir(dataset_path):

    actor_path = os.path.join(dataset_path, actor)

    # Only process Actor folders
    if actor.startswith("Actor_"):

        for file in os.listdir(actor_path):

            if file.endswith(".wav"):

                emotion_code = file.split("-")[2]

                paths.append(os.path.join(actor_path, file))
                emotions.append(emotion_dict[emotion_code])


In [6]:
df = pd.DataFrame({
    "Path": paths,
    "Emotion": emotions
})

df.head()

,Path,Emotion
0,C:\Users\saniy\Downloads\code_Alpha_T2\Actor_0...,neutral
1,C:\Users\saniy\Downloads\code_Alpha_T2\Actor_0...,neutral
2,C:\Users\saniy\Downloads\code_Alpha_T2\Actor_0...,neutral
3,C:\Users\saniy\Downloads\code_Alpha_T2\Actor_0...,neutral
4,C:\Users\saniy\Downloads\code_Alpha_T2\Actor_0...,calm


In [7]:
#Function to extract MFCC features from an audio file (Audio -> raw numbers)
def extract_features(file_path):

    #Loading audiofile (converting the audio file into a numerical representation to understand ML model)
    audio, sample_rate= librosa.load(file_path, duration=3, offset=0.5) # Load the All files equally to prevent bias in the model.

    #Extracting the important features from the audio file
    mfcc= librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40) 

    # Take mean of the MFCC feature to get a feature vector
    mfcc_mean = np.mean(mfcc.T, axis=0)

    return mfcc_mean


In [8]:
#raw numbers -> features
X= []

for path in df["Path"]:
    features = extract_features(path)
    X.append(features)



In [9]:
X = np.array(X)

print(X.shape)

(1440, 40)


In [10]:
y = df["Emotion"] 

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
encoder = LabelEncoder()
y= encoder.fit_transform(y)

In [13]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [14]:
print(X_train.shape)
print(X_test.shape)

(1152, 40)
(288, 40)


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


In [ ]:
model = Sequential()

model.add(LSTM(128, input_shape=(40,1))) #128 is the neuron units(neuron memory cells) and 40 is the features and 1 is a time frame to chop the audio into small pieces
model.add(Dropout(0.3)) #Dropping 30% of the neurons to prevent overfitting for the model
model.add(Dense(65, activation='relu')) #Dense is a fully connected layer every neuron is connected to previous layer so there are 65 neurons each learning a different aspect of input and passes a result through a relu 
model.add(Dense(8, activation= 'softmax')) #8 is the emotions we are trying to classify and softmax is used to get the probability of each emotion


c:\Users\saniy\Downloads\code_Alpha_T2\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [18]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [19]:
#fit is a training func in keras , epoch = model will see the whole dataset 50 times , split data into 32 samples (Training data), after each epoch it returns the validation accuracy and loss
#An epoch is completed by processing the training data one mini-batch at a time until the entire dataset has been covered once.
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.1556 - loss: 2.0665 - val_accuracy: 0.2049 - val_loss: 2.0043
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.2210 - loss: 1.9831 - val_accuracy: 0.2604 - val_loss: 1.8797
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.2754 - loss: 1.9095 - val_accuracy: 0.2708 - val_loss: 1.8554
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.2739 - loss: 1.8464 - val_accuracy: 0.2604 - val_loss: 1.8592
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3022 - loss: 1.8118 - val_accuracy: 0.2812 - val_loss: 1.8138
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.2952 - loss: 1.7909 - val_accuracy: 0.3299 - val_loss: 1.7696
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3181 - loss: 1.7791 - val_accuracy: 0.3333 - val_loss: 1.7753
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.3457 - loss: 1.7126 - val_accuracy: 0.3403 - v

In [33]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3878 - loss: 1.7734
Test Accuracy: 0.3854166567325592


In [37]:
model.save("emotion_model.h5")

In [1]:
import joblib
joblib.dump(encoder, "label_encodder.pkl")

NameError: name 'encoder' is not defined